In [3]:
import mujoco
import numpy as np
from scipy.spatial.transform import Rotation as R

def get_mjcf_offset_official(xml_path, base_name, tool_name):
    """
    使用 MuJoCo 官方库计算 XML 中两个 Body 之间的相对位姿偏移
    """
    try:
        # 1. 加载模型
        model = mujoco.MjModel.from_xml_path(xml_path)
        data = mujoco.MjData(model)

        # 2. 执行前向动力学计算，刷新所有物体的空间位置
        mujoco.mj_forward(model, data)

        # 3. 获取 Body 的 ID
        # 注意：如果你的 afp_roll_link 是 site，请将 mjOBJ_BODY 改为 mjOBJ_SITE
        try:
            base_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, base_name)
            tool_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, tool_name)
        except:
            # 自动尝试作为 site 查找
            base_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, base_name)
            tool_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, tool_name)

        if base_id == -1 or tool_id == -1:
            raise ValueError(f"找不到名称: {base_name} 或 {tool_name}，请检查 XML 中的命名")

        # 4. 获取世界坐标系 (World) 下的位姿
        # xpos: 位置 (3,), xmat: 旋转矩阵 (3x3)
        pos_base = data.xpos[base_id]
        mat_base = data.xmat[base_id].reshape(3, 3)

        pos_tool = data.xpos[tool_id]
        mat_tool = data.xmat[tool_id].reshape(3, 3)

        # 5. 计算相对位姿 (坐标变换)
        # 相对平移：在 base 坐标系下的表达
        # rel_p = R_base^T * (p_tool - p_base)
        rel_pos = mat_base.T @ (pos_tool - pos_base)

        # 相对旋转：R_rel = R_base^T * R_tool
        rel_mat = mat_base.T @ mat_tool

        return rel_pos, rel_mat

    except Exception as e:
        print(f"MuJoCo 解析失败: {e}")
        return None, None

if __name__ == "__main__":
    XML_FILE = "/home/lgx/Project/AFP/src/afp_mjc/env/mujoco_ur5e/ur5e.xml"
    BASE = "flange"
    TOOL = "afp_roll_link"

    pos, mat = get_mjcf_offset_official(XML_FILE, BASE, TOOL)

    if pos is not None:
        print("-" * 40)
        print(f"工具末端 [{TOOL}] 相对于法兰 [{BASE}] 的偏移:")
        print("-" * 40)
        print(f"平移 Translation (x, y, z) [m]:\n{pos}")
        
        # 转换为四元数
        r = R.from_matrix(mat)
        quat = r.as_quat() # x, y, z, w
        print(f"\n四元数 Quaternion (x, y, z, w):\n{quat}")
        
        # 转换为欧拉角
        euler = r.as_euler('xyz', degrees=True)
        print(f"\n欧拉角 Euler (Roll, Pitch, Yaw) [deg]:\n{euler}")
        print("-" * 40)

        # 格式化输出，方便直接复制到代码里
        print(f"\n代码配置建议:")
        print(f"TOOL_OFFSET_POS = {pos.tolist()}")
        print(f"TOOL_OFFSET_QUAT = {quat.tolist()}")

----------------------------------------
工具末端 [afp_roll_link] 相对于法兰 [flange] 的偏移:
----------------------------------------
平移 Translation (x, y, z) [m]:
[0.    0.    0.161]

四元数 Quaternion (x, y, z, w):
[0. 0. 0. 1.]

欧拉角 Euler (Roll, Pitch, Yaw) [deg]:
[0. 0. 0.]
----------------------------------------

代码配置建议:
TOOL_OFFSET_POS = [0.0, 0.0, 0.16099999999999987]
TOOL_OFFSET_QUAT = [0.0, 0.0, 0.0, 1.0]


In [5]:
"""
末端执行器位姿对比：Pinocchio (URDF) vs MuJoCo (XML)
两者均计算末端在机器人基座坐标系下的位置和姿态
"""

import numpy as np
import pinocchio as pin
import mujoco
from scipy.spatial.transform import Rotation


# ─────────────────────────────────────────────
# 配置：根据你的实际情况修改以下参数
# ─────────────────────────────────────────────
URDF_PATH       = "/home/lgx/Project/AFP/src/afp_mjc/env/mujoco_ur5e/ur5e.urdf"          # URDF 文件路径
MJCF_PATH       = "/home/lgx/Project/AFP/src/afp_mjc/env/mujoco_ur5e/ur5e.xml"           # MuJoCo XML 文件路径
EE_FRAME_NAME   = "flange"             # 末端执行器 frame/body 名称
BASE_FRAME_NAME_XML = "base"
BASE_FRAME_NAME_URDF = "base_link"           # 机器人基座 frame/body 名称

# 关节角度（弧度），按模型关节顺序填写
JOINT_ANGLES = np.array([0.0, -0.5, 0.5, -1.0, 0.0, 1.0])
# ─────────────────────────────────────────────


def mat_to_quat_xyzw(R: np.ndarray) -> np.ndarray:
    """旋转矩阵 → 四元数 [x, y, z, w]"""
    return Rotation.from_matrix(R).as_quat()


def quat_wxyz_to_xyzw(q: np.ndarray) -> np.ndarray:
    """MuJoCo/Pinocchio [w, x, y, z] → ROS/scipy [x, y, z, w]"""
    return np.array([q[1], q[2], q[3], q[0]])


# ═══════════════════════════════════════════════════════════
# 1. Pinocchio：基于 URDF
# ═══════════════════════════════════════════════════════════
def compute_pose_pinocchio(urdf_path, ee_frame_name, base_frame_name, q):
    """
    使用 Pinocchio 计算末端在基座坐标系下的位姿
    返回: (position [3], quaternion_xyzw [4])
    """
    # 加载模型
    model = pin.buildModelFromUrdf(urdf_path)
    data  = model.createData()

    # 确认 frame 存在
    if not model.existFrame(ee_frame_name):
        raise ValueError(f"[Pinocchio] Frame '{ee_frame_name}' not found. "
                         f"Available: {[model.frames[i].name for i in range(model.nframes)]}")
    if not model.existFrame(base_frame_name):
        raise ValueError(f"[Pinocchio] Frame '{base_frame_name}' not found.")

    ee_frame_id   = model.getFrameId(ee_frame_name)
    base_frame_id = model.getFrameId(base_frame_name)

    # 前向运动学
    pin.forwardKinematics(model, data, q)
    pin.updateFramePlacements(model, data)

    # 末端和基座在世界系下的变换 (SE3)
    T_world_ee   = data.oMf[ee_frame_id]
    T_world_base = data.oMf[base_frame_id]

    # 末端相对于基座的变换：T_base_ee = T_world_base^{-1} * T_world_ee
    T_base_ee = T_world_base.inverse() * T_world_ee

    pos  = T_base_ee.translation                        # [x, y, z]
    quat = mat_to_quat_xyzw(T_base_ee.rotation)        # [x, y, z, w]

    return pos, quat, model, data, T_base_ee


# ═══════════════════════════════════════════════════════════
# 2. MuJoCo：基于 XML
# ═══════════════════════════════════════════════════════════
def compute_pose_mujoco(xml_path, ee_body_name, base_body_name, joint_angles):
    """
    使用 MuJoCo 计算末端在基座坐标系下的位姿
    返回: (position [3], quaternion_xyzw [4])
    """
    model = mujoco.MjModel.from_xml_path(xml_path)
    data  = mujoco.MjData(model)

    # 设置关节角度（只设置 actuated joints，跳过 free/ball joints）
    actuated_joint_ids = []
    for i in range(model.njnt):
        jtype = model.jnt_type[i]
        # 只处理 hinge/slide 关节 (type 2/3)
        if jtype in (mujoco.mjtJoint.mjJNT_HINGE, mujoco.mjtJoint.mjJNT_SLIDE):
            actuated_joint_ids.append(i)

    if len(joint_angles) != len(actuated_joint_ids):
        print(f"[MuJoCo] Warning: {len(joint_angles)} angles provided "
              f"but {len(actuated_joint_ids)} hinge/slide joints found. "
              f"Setting available joints.")

    for idx, jnt_id in enumerate(actuated_joint_ids):
        if idx >= len(joint_angles):
            break
        qadr = model.jnt_qposadr[jnt_id]
        data.qpos[qadr] = joint_angles[idx]

    # 前向运动学
    mujoco.mj_kinematics(model, data)

    # 获取 body id
    ee_id   = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, ee_body_name)
    base_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, base_body_name)

    if ee_id == -1:
        all_bodies = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i)
                      for i in range(model.nbody)]
        raise ValueError(f"[MuJoCo] Body '{ee_body_name}' not found. "
                         f"Available: {all_bodies}")
    if base_id == -1:
        raise ValueError(f"[MuJoCo] Body '{base_body_name}' not found.")

    # 末端和基座在世界系下的位姿
    pos_ee_world   = data.xpos[ee_id].copy()
    R_world_ee     = data.xmat[ee_id].reshape(3, 3).copy()

    pos_base_world = data.xpos[base_id].copy()
    R_world_base   = data.xmat[base_id].reshape(3, 3).copy()

    # 变换到基座坐标系
    # p_base = R_world_base^T * (p_ee_world - p_base_world)
    pos_in_base = R_world_base.T @ (pos_ee_world - pos_base_world)

    # R_base_ee = R_world_base^T * R_world_ee
    R_in_base   = R_world_base.T @ R_world_ee
    quat        = mat_to_quat_xyzw(R_in_base)   # [x, y, z, w]

    return pos_in_base, quat, model, data


# ═══════════════════════════════════════════════════════════
# 3. 打印与对比
# ═══════════════════════════════════════════════════════════
def print_pose(label, pos, quat_xyzw):
    """格式化打印位姿"""
    rpy = Rotation.from_quat(quat_xyzw).as_euler('xyz', degrees=True)
    print(f"\n{'─'*50}")
    print(f"  {label}")
    print(f"{'─'*50}")
    print(f"  Position  (x, y, z)  [m]  : "
          f"{pos[0]:+.6f}  {pos[1]:+.6f}  {pos[2]:+.6f}")
    print(f"  Quaternion (x,y,z,w)      : "
          f"{quat_xyzw[0]:+.6f}  {quat_xyzw[1]:+.6f}  "
          f"{quat_xyzw[2]:+.6f}  {quat_xyzw[3]:+.6f}")
    print(f"  RPY (roll,pitch,yaw) [deg]: "
          f"{rpy[0]:+.4f}  {rpy[1]:+.4f}  {rpy[2]:+.4f}")


def compare_poses(pos_pin, quat_pin, pos_mjc, quat_mjc):
    """对比两者差异"""
    d_pos  = np.linalg.norm(pos_pin - pos_mjc)

    # 四元数误差（取最小旋转角）
    R_pin  = Rotation.from_quat(quat_pin)
    R_mjc  = Rotation.from_quat(quat_mjc)
    R_diff = R_pin.inv() * R_mjc
    angle_diff = np.abs(R_diff.magnitude()) * 180.0 / np.pi

    print(f"\n{'═'*50}")
    print(f"  对比结果")
    print(f"{'═'*50}")
    print(f"  位置误差 (L2 norm) [m]  : {d_pos:.6e}")
    print(f"  旋转误差 (angle)  [deg] : {angle_diff:.6e}")

    if d_pos < 1e-4 and angle_diff < 1e-3:
        print("  ✅ 两者结果一致！")
    else:
        print("  ⚠️  存在差异，请检查模型对齐或关节顺序")


# ═══════════════════════════════════════════════════════════
# 主程序
# ═══════════════════════════════════════════════════════════
if __name__ == "__main__":
    print("=" * 50)
    print("  末端位姿对比：Pinocchio vs MuJoCo")
    print(f"  EE   frame : {EE_FRAME_NAME}")
    print(f"  Base frame : {BASE_FRAME_NAME_XML} (XML), {BASE_FRAME_NAME_URDF} (URDF)")
    print(f"  Joint q    : {JOINT_ANGLES}")
    print("=" * 50)

    # Pinocchio
    try:
        pos_pin, quat_pin, *_ = compute_pose_pinocchio(
            URDF_PATH, EE_FRAME_NAME, BASE_FRAME_NAME_URDF, JOINT_ANGLES)
        print_pose("Pinocchio (URDF)", pos_pin, quat_pin)
    except Exception as e:
        print(f"\n[Pinocchio] 错误: {e}")
        pos_pin, quat_pin = None, None

    # MuJoCo
    try:
        pos_mjc, quat_mjc, *_ = compute_pose_mujoco(
            MJCF_PATH, EE_FRAME_NAME, BASE_FRAME_NAME_XML, JOINT_ANGLES)
        print_pose("MuJoCo   (XML) ", pos_mjc, quat_mjc)
    except Exception as e:
        print(f"\n[MuJoCo] 错误: {e}")
        pos_mjc, quat_mjc = None, None

    # 对比
    if pos_pin is not None and pos_mjc is not None:
        compare_poses(pos_pin, quat_pin, pos_mjc, quat_mjc)

    print()

  末端位姿对比：Pinocchio vs MuJoCo
  EE   frame : flange
  Base frame : base (XML), base_link (URDF)
  Joint q    : [ 0.  -0.5  0.5 -1.   0.   1. ]

──────────────────────────────────────────────────
  Pinocchio (URDF)
──────────────────────────────────────────────────
  Position  (x, y, z)  [m]  : -0.848877  -0.234914  +0.312103
  Quaternion (x,y,z,w)      : +0.707972  +0.000776  +0.000780  +0.706239
  RPY (roll,pitch,yaw) [deg]: +90.1404  -0.0005  +0.1260

──────────────────────────────────────────────────
  MuJoCo   (XML) 
──────────────────────────────────────────────────
  Position  (x, y, z)  [m]  : +0.849120  +0.134000  +0.312726
  Quaternion (x,y,z,w)      : +0.000000  +0.707107  +0.707107  -0.000000
  RPY (roll,pitch,yaw) [deg]: +90.0000  +0.0000  -180.0000

══════════════════════════════════════════════════
  对比结果
══════════════════════════════════════════════════
  位置误差 (L2 norm) [m]  : 1.737611e+00
  旋转误差 (angle)  [deg] : 1.798740e+02
  ⚠️  存在差异，请检查模型对齐或关节顺序

